## Read CSV file

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
import tempfile

### Reading CSV file

In [3]:
df = pd.read_csv("data_censored.csv")
df.head(20)

,id,period,treatment,x1,x2,x3,x4,age,age_s,outcome,censored,eligible
0,1,0,1,1,1.146148,0,0.734203,36,0.083333,0,0,1
1,1,1,1,1,0.002200,0,0.734203,37,0.166667,0,0,0
2,1,2,1,0,-0.481762,0,0.734203,38,0.250000,0,0,0
3,1,3,1,0,0.007872,0,0.734203,39,0.333333,0,0,0
4,1,4,1,1,0.216054,0,0.734203,40,0.416667,0,0,0
5,1,5,1,0,-0.057482,0,0.734203,41,0.500000,0,1,0
6,2,0,0,1,-0.802142,0,-0.990794,26,-0.750000,0,0,1
7,2,1,1,1,-0.983030,0,-0.990794,27,-0.666667,0,0,1
8,2,2,1,0,0.399388,0,-0.990794,28,-0.583333,0,0,0
9,2,3,0,0,1.835085,0,-0.990794,29,-0.500000,0,0,0


### Function Definitions

In [4]:
class TrialSequence:
    def __init__(self, estimand):
        self.estimand = estimand
        self.data = None
        self.id_col = None
        self.period_col = None
        self.treatment_col = None
        self.outcome_col = None
        self.eligible_col = None
        
        # To hold weight model settings and fitted models
        self.switch_weights = None   # Only for PP; not used for ITT.
        self.censor_weights = None

    def set_data(self, data, id_col, period_col, treatment_col, outcome_col, eligible_col):
        """
        Mimics the R set_data() function: stores the data and key column names.
        """
        self.data = data
        self.id_col = id_col
        self.period_col = period_col
        self.treatment_col = treatment_col
        self.outcome_col = outcome_col
        self.eligible_col = eligible_col
        return self

    def set_switch_weight_model(self, numerator, denominator, model_fitter, save_path=None):
        """
        Sets the switch weight model.
        The formulas are provided as strings (e.g., "age" or "age + x1 + x3").
        Internally, the full model formulas will be constructed using the treatment column.
        """
        self.switch_weights = {
            "numerator_formula": f"{self.treatment_col} ~ {numerator}",
            "denom_formula": f"{self.treatment_col} ~ {denominator}",
            "model_fitter": model_fitter,
            "save_path": save_path,
            "fitted_numerator": None,
            "fitted_denominator": None,
            "weights": None
        }
        return self

    def set_censor_weight_model(self, censor_event, numerator, denominator, pool_models, model_fitter, save_path=None):
        """
        Sets the censor weight model.
        In the R code the formulas are defined as:
          Numerator: 1 - <censor_event> ~ numerator
          Denom:     1 - <censor_event> ~ denominator
        The pool_models argument indicates whether the numerator is pooled ("numerator") or not.
        """
        self.censor_weights = {
            "censor_event": censor_event,
            "numerator_formula": f"1 - {censor_event} ~ {numerator}",
            "denom_formula": f"1 - {censor_event} ~ {denominator}",
            "pool_models": pool_models,
            "model_fitter": model_fitter,
            "save_path": save_path,
            "fitted_numerator": None,
            "fitted_denominator": None,
            "weights": None
        }
        return self

    def calculate_weights(self):
        """
        Fits the logistic regression models for each weight model and computes stabilized weights.
        For each weight type, the weight is defined as the ratio of the predicted probability from the
        numerator model to that from the denominator model.
        """
        # Calculate switch weights (only for PP)
        if self.switch_weights is not None:
            sw = self.switch_weights
            # Construct save paths if provided
            num_save = os.path.join(sw["save_path"], "switch_numerator.pkl") if sw["save_path"] else None
            den_save = os.path.join(sw["save_path"], "switch_denom.pkl") if sw["save_path"] else None
            sw["fitted_numerator"] = sw["model_fitter"](sw["numerator_formula"], data=self.data, save_path=num_save)
            sw["fitted_denominator"] = sw["model_fitter"](sw["denom_formula"], data=self.data, save_path=den_save)
            # Calculate weight as the ratio of predictions
            self.data["switch_weight"] = sw["fitted_numerator"].predict(self.data) / sw["fitted_denominator"].predict(self.data)
            sw["weights"] = self.data["switch_weight"]

        # Calculate censor weights (for both PP and ITT)
        if self.censor_weights is not None:
            cw = self.censor_weights
            num_save = os.path.join(cw["save_path"], "censor_numerator.pkl") if cw["save_path"] else None
            den_save = os.path.join(cw["save_path"], "censor_denom.pkl") if cw["save_path"] else None
            cw["fitted_numerator"] = cw["model_fitter"](cw["numerator_formula"], data=self.data, save_path=num_save)
            cw["fitted_denominator"] = cw["model_fitter"](cw["denom_formula"], data=self.data, save_path=den_save)
            self.data["censor_weight"] = cw["fitted_numerator"].predict(self.data) / cw["fitted_denominator"].predict(self.data)
            cw["weights"] = self.data["censor_weight"]

        return self

    def show_weight_models(self):
        """
        Prints a summary of the weight models stored in the object.
        """
        if self.switch_weights is not None:
            sw = self.switch_weights
            print("Switch Weights Model:")
            print("  Numerator formula:", sw["numerator_formula"])
            print("  Denominator formula:", sw["denom_formula"])
            print("  Model fitter type:", sw["model_fitter"].__name__)
            if sw["fitted_numerator"] is None:
                print("  Weight models not fitted. Use calculate_weights().")
            else:
                print("  Weights calculated.")
        if self.censor_weights is not None:
            cw = self.censor_weights
            print("Censor Weights Model:")
            print("  Numerator formula:", cw["numerator_formula"])
            print("  Denominator formula:", cw["denom_formula"])
            print("  Pooling:", cw["pool_models"])
            print("  Model fitter type:", cw["model_fitter"].__name__)
            if cw["fitted_numerator"] is None:
                print("  Weight models not fitted. Use calculate_weights().")
            else:
                print("  Weights calculated.")



### Setup

In [5]:
trial_pp = TrialSequence(estimand="PP")

trial_pp.set_data(
    data=df,
    id_col='id',
    period_col='period',
    treatment_col='treatment',
    outcome_col='outcome',
    eligible_col='eligible'
)

trial_itt = TrialSequence(estimand="ITT")

trial_itt.set_data(
    data=df,
    id_col='id',
    period_col='period',
    treatment_col='treatment',
    outcome_col='outcome',
    eligible_col='eligible'
)

In [6]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pickle

def stats_glm_logit(formula, data, save_path=None):
    # Use sm.families.Binomial() instead of smf.families.Binomial()
    model = smf.glm(formula=formula, data=data, family=sm.families.Binomial())
    result = model.fit()
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(result, f)
    return result

def te_stats_glm_logit(save_path=None):
    def fit_model(formula, data, model_type='weights', weights=None, cluster_var='id'):
        if model_type == 'weights':
            # Fit a standard logistic regression model.
            model = smf.glm(formula=formula, data=data, family=sm.families.Binomial())
            result = model.fit()
        elif model_type == 'outcome':
            # Copy data and assign weights (default to 1 if not provided)
            data = data.copy()
            data['weights'] = weights if weights is not None else 1.0
            model = smf.glm(formula=formula, data=data, family=sm.families.Binomial(), weights=data['weights'])
            # Fit using cluster robust standard errors
            result = model.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_var]})
        else:
            raise ValueError("Unknown model_type. Use 'weights' or 'outcome'.")
        
        # Optionally save the fitted model
        if save_path:
            os.makedirs(save_path, exist_ok=True)
            file_path = os.path.join(save_path, f"model_{model_type}.pkl")
            with open(file_path, 'wb') as f:
                pickle.dump(result, f)
        
        return result

    return fit_model


In [7]:
print("trial pp before switch weighs")
print(trial_pp.__dict__)

trial pp before switch weighs
{'estimand': 'PP', 'data':      id  period  treatment  x1        x2  x3        x4  age     age_s  \
0     1       0          1   1  1.146148   0  0.734203   36  0.083333   
1     1       1          1   1  0.002200   0  0.734203   37  0.166667   
2     1       2          1   0 -0.481762   0  0.734203   38  0.250000   
3     1       3          1   0  0.007872   0  0.734203   39  0.333333   
4     1       4          1   1  0.216054   0  0.734203   40  0.416667   
..   ..     ...        ...  ..       ...  ..       ...  ...       ...   
720  99       3          0   0 -0.747906   1  0.575268   68  2.750000   
721  99       4          0   0 -0.790056   1  0.575268   69  2.833333   
722  99       5          1   1  0.387429   1  0.575268   70  2.916667   
723  99       6          1   1 -0.033762   1  0.575268   71  3.000000   
724  99       7          0   0 -1.340497   1  0.575268   72  3.083333   

     outcome  censored  eligible  
0          0         0         

## 3. Weight models and censoring

### 3.1 Censoring due to treatment switching


In [8]:
trial_pp.set_switch_weight_model(
    numerator="age",  # use the column names as strings
    denominator="age + x1 + x3",
    model_fitter=stats_glm_logit  # your logistic regression model fitting function
)

trial_pp.calculate_weights()

In [9]:
print("trial pp after switch weighs")
print(trial_pp.__dict__)

trial pp after switch weighs
{'estimand': 'PP', 'data':      id  period  treatment  x1        x2  x3        x4  age     age_s  \
0     1       0          1   1  1.146148   0  0.734203   36  0.083333   
1     1       1          1   1  0.002200   0  0.734203   37  0.166667   
2     1       2          1   0 -0.481762   0  0.734203   38  0.250000   
3     1       3          1   0  0.007872   0  0.734203   39  0.333333   
4     1       4          1   1  0.216054   0  0.734203   40  0.416667   
..   ..     ...        ...  ..       ...  ..       ...  ...       ...   
720  99       3          0   0 -0.747906   1  0.575268   68  2.750000   
721  99       4          0   0 -0.790056   1  0.575268   69  2.833333   
722  99       5          1   1  0.387429   1  0.575268   70  2.916667   
723  99       6          1   1 -0.033762   1  0.575268   71  3.000000   
724  99       7          0   0 -1.340497   1  0.575268   72  3.083333   

     outcome  censored  eligible  switch_weight  
0          0     

### 3.2 Other informative censoring

In [10]:
trial_pp.set_censor_weight_model(
    censor_event = "censored",
    numerator    = "x2",
    denominator  = "x2 + x1",
    pool_models  = "none",
    model_fitter= stats_glm_logit  # your logistic regression model fitting function
)

trial_pp.calculate_weights()

d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSep

In [11]:
print("trial pp after censor weighs")
print(trial_pp.__dict__)
print("Value counts for censor_weight:")
print(trial_pp.data["censor_weight"].value_counts())


trial pp after censor weighs
{'estimand': 'PP', 'data':      id  period  treatment  x1        x2  x3        x4  age     age_s  \
0     1       0          1   1  1.146148   0  0.734203   36  0.083333   
1     1       1          1   1  0.002200   0  0.734203   37  0.166667   
2     1       2          1   0 -0.481762   0  0.734203   38  0.250000   
3     1       3          1   0  0.007872   0  0.734203   39  0.333333   
4     1       4          1   1  0.216054   0  0.734203   40  0.416667   
..   ..     ...        ...  ..       ...  ..       ...  ...       ...   
720  99       3          0   0 -0.747906   1  0.575268   68  2.750000   
721  99       4          0   0 -0.790056   1  0.575268   69  2.833333   
722  99       5          1   1  0.387429   1  0.575268   70  2.916667   
723  99       6          1   1 -0.033762   1  0.575268   71  3.000000   
724  99       7          0   0 -1.340497   1  0.575268   72  3.083333   

     outcome  censored  eligible  switch_weight  censor_weight  
0 

In [12]:
trial_itt.set_censor_weight_model(
    censor_event = "censored",
    numerator    = "x2",
    denominator  = "x2 + x1",
    pool_models  = "none",
    model_fitter=stats_glm_logit  # your logistic regression model fitting function
)

trial_itt.calculate_weights()

d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
d:\Anaconda\envs\data_ana\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSep

In [13]:
def calculate_ipw(df, treatment_col='treatment', propensity_col='propensity'):
    """
    Compute IPW where:
      - For treated: weight = 1/propensity
      - For controls: weight = 1/(1 - propensity)
    """
    treated = df[treatment_col] == 1
    control = df[treatment_col] == 0
    df['ipw'] = np.where(treated, 1 / df[propensity_col], 1 / (1 - df[propensity_col]))
    return df

In [20]:
print(trial_pp)
df_trial_pp = pd.DataFrame(list(trial_pp))
df_trial_pp = calculate_ipw(df_trial_pp)


TypeError: 'TrialSequence' object is not iterable

### Mimicking trial sequence

In [ ]:
def trial_sequence(estimand):
    # In reality, you'd have logic to set up the 'trial_sequence'
    return {"estimand": estimand, "some_data": None}

# Create two trial-sequence-like objects
trial_pp  = trial_sequence("PP")   # per-protocol
trial_itt = trial_sequence("ITT")  # intention-to-treat

# Build directory paths in the system’s temp folder
trial_pp_dir  = os.path.join(tempfile.gettempdir(), "trial_pp")
trial_itt_dir = os.path.join(tempfile.gettempdir(), "trial_itt")

os.makedirs(trial_pp_dir, exist_ok=True)
os.makedirs(trial_itt_dir, exist_ok=True)

In [ ]:
print("trial_pp:", trial_pp)
print("trial_itt:", trial_itt)
print("Directories created:", trial_pp_dir, trial_itt_dir)


trial_pp: {'estimand': 'PP', 'some_data': None}
trial_itt: {'estimand': 'ITT', 'some_data': None}
Directories created: C:\Users\Alex\AppData\Local\Temp\trial_pp C:\Users\Alex\AppData\Local\Temp\trial_itt


## Data prep

In [ ]:
def stats_glm_logit(formula, data, save_path=None):
    """
    Fits a logistic regression model using statsmodels.
    Optionally saves the fitted model to disk.
    """
    model = smf.logit(formula, data=data).fit(disp=0)
    if save_path is not None:
        with open(save_path, "wb") as f:
            pickle.dump(model, f)
    return model

In [ ]:
# Create temporary directories (like trial_pp_dir and trial_itt_dir)
trial_pp_dir = os.path.join(tempfile.gettempdir(), "trial_pp")
trial_itt_dir = os.path.join(tempfile.gettempdir(), "trial_itt")
os.makedirs(trial_pp_dir, exist_ok=True)
os.makedirs(trial_itt_dir, exist_ok=True)


In [ ]:
trial_pp = TrialSequence("PP").set_data(
    data=df,
    id_col="id",
    period_col="period",
    treatment_col="treatment",
    outcome_col="outcome",
    eligible_col="eligible"
)

trial_itt = TrialSequence("ITT").set_data(
    data=df,
    id_col="id",
    period_col="period",
    treatment_col="treatment",
    outcome_col="outcome",
    eligible_col="eligible"
)


In [ ]:
print("trial_itt contents:")
print(trial_itt.__dict__)

print("\ntrial_pp contents:")
print(trial_pp.__dict__)


trial_itt contents:
{'estimand': 'ITT', 'data':      id  period  treatment  x1        x2  x3        x4  age     age_s  \
0     1       0          1   1  1.146148   0  0.734203   36  0.083333   
1     1       1          1   1  0.002200   0  0.734203   37  0.166667   
2     1       2          1   0 -0.481762   0  0.734203   38  0.250000   
3     1       3          1   0  0.007872   0  0.734203   39  0.333333   
4     1       4          1   1  0.216054   0  0.734203   40  0.416667   
..   ..     ...        ...  ..       ...  ..       ...  ...       ...   
720  99       3          0   0 -0.747906   1  0.575268   68  2.750000   
721  99       4          0   0 -0.790056   1  0.575268   69  2.833333   
722  99       5          1   1  0.387429   1  0.575268   70  2.916667   
723  99       6          1   1 -0.033762   1  0.575268   71  3.000000   
724  99       7          0   0 -1.340497   1  0.575268   72  3.083333   

     outcome  censored  eligible  switch_weight  censor_weight  
0         

### 3.1 Censoring due to treatment switching (only for PP)

In [ ]:
import os
from functools import partial

# Assume trial_pp is an instance of TrialSequence and trial_pp_dir is defined.
# Create a "partial" function so that stats_glm_logit is pre-configured with a save_path.
switch_model_fitter = partial(stats_glm_logit, save_path=os.path.join(trial_pp_dir, "switch_models"))

# Set the switch weight model on trial_pp. This is equivalent to the R code:
# trial_pp <- trial_pp |> set_switch_weight_model(
#   numerator    = ~ age,
#   denominator  = ~ age + x1 + x3,
#   model_fitter = stats_glm_logit(save_path = file.path(trial_pp_dir, "switch_models"))
# )
trial_pp.set_switch_weight_model(
    numerator="age",
    denominator="age + x1 + x3",
    model_fitter=switch_model_fitter
)

# To view the switch weight settings (equivalent to trial_pp@switch_weights in R):
print("Switch weights settings for trial_pp:")
print(trial_pp.switch_weights)


Switch weights settings for trial_pp:
{'numerator_formula': 'treatment ~ age', 'denom_formula': 'treatment ~ age + x1 + x3', 'model_fitter': functools.partial(<function stats_glm_logit at 0x000002FA5ABD3880>, save_path='C:\\Users\\Alex\\AppData\\Local\\Temp\\trial_pp\\switch_models'), 'save_path': None, 'fitted_numerator': None, 'fitted_denominator': None, 'weights': None}


In [ ]:
trial_pp.calculate_weights()

In [ ]:
import os
from functools import partial

# Create a partial function for PP censor weight model fitter:
censor_model_fitter_pp = partial(stats_glm_logit, save_path=os.path.join(trial_pp_dir, "switch_models"))
# Set censor weight model for trial_pp (PP)
trial_pp.set_censor_weight_model(
    censor_event="censored",
    numerator="x2",         # This will yield: 1 - censored ~ x2
    denominator="x2 + x1",  # This will yield: 1 - censored ~ x2 + x1
    pool_models="none",
    model_fitter=censor_model_fitter_pp,
    save_path=trial_pp_dir
)

trial_pp.censor_weights()

NameError: name 'trial_pp_dir' is not defined

c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detect

LinAlgError: Singular matrix

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from patsy import dmatrices

# ---------------------------------------------------
# 1. SETUP: Define estimands and create directories
# ---------------------------------------------------
estimand_pp = "PP"   # Per-protocol
estimand_itt = "ITT" # Intention-to-treat

trial_pp_dir = os.path.join(tempfile.gettempdir(), "trial_pp")
os.makedirs(trial_pp_dir, exist_ok=True)

trial_itt_dir = os.path.join(tempfile.gettempdir(), "trial_itt")
os.makedirs(trial_itt_dir, exist_ok=True)

# ---------------------------------------------------
# 2. DATA PREPARATION
# ---------------------------------------------------
# For demonstration, we simulate a dummy dataset similar to "data_censored"
np.random.seed(42)
n_patients = 3
n_periods = 6
data_censored = pd.read_csv("data_censored.csv")

print("Sample of data_censored:")
print(data_censored.head())

# Save the trial configuration as dictionaries
trial_pp = {'estimand': estimand_pp, 'data': data_censored.copy(), 'dir': trial_pp_dir}
trial_itt = {'estimand': estimand_itt, 'data': data_censored.copy(), 'dir': trial_itt_dir}

# ---------------------------------------------------
# 3. WEIGHT MODELS AND CENSORING
# ---------------------------------------------------
# Define a helper to fit logistic regression via statsmodels
def fit_logistic_model(formula, data):
    y, X = dmatrices(formula, data, return_type='dataframe')
    model = sm.Logit(y, X).fit(disp=False)
    return model

# For treatment switching (only applicable for PP):
#   - Numerator: treatment ~ age
#   - Denominator: treatment ~ age + x1 + x3
#
# For a “previous treatment” variable, we simulate it as the lagged treatment for each patient.
data_censored['prev_treatment'] = data_censored.groupby('id')['treatment'].shift(1)
data_censored['prev_treatment'].fillna(data_censored['treatment'], inplace=True)

# Fit models for patients with previous treatment == 1
mask_prev1 = data_censored['prev_treatment'] == 1
num_model_switch = fit_logistic_model("treatment ~ age", data_censored[mask_prev1])
den_model_switch = fit_logistic_model("treatment ~ age + x1 + x3", data_censored[mask_prev1])

# Predict probabilities for switching weights
data_censored.loc[mask_prev1, 'p_num'] = num_model_switch.predict(sm.add_constant(
    data_censored.loc[mask_prev1, 'age']
))
data_censored.loc[mask_prev1, 'p_den'] = den_model_switch.predict(
    sm.add_constant(data_censored.loc[mask_prev1, ['age', 'x1', 'x3']])
)

# Stabilized weight for treatment switching (for PP)
data_censored['switch_weight'] = np.nan
data_censored.loc[mask_prev1, 'switch_weight'] = (
    data_censored.loc[mask_prev1, 'p_num'] / data_censored.loc[mask_prev1, 'p_den']
)

# For informative censoring (both PP and ITT):
#   - Use transformed outcome: not_censored = 1 - censored
data_censored['not_censored'] = 1 - data_censored['censored']

# For PP:
#   - Numerator: not_censored ~ x2
#   - Denominator: not_censored ~ x2 + x1
num_model_censor_pp = fit_logistic_model("not_censored ~ x2", data_censored)
den_model_censor_pp = fit_logistic_model("not_censored ~ x2 + x1", data_censored)

data_censored['p_num_cens'] = num_model_censor_pp.predict(sm.add_constant(data_censored['x2']))
data_censored['p_den_cens'] = den_model_censor_pp.predict(sm.add_constant(data_censored[['x2', 'x1']]))

data_censored['censor_weight'] = data_censored['p_num_cens'] / data_censored['p_den_cens']

# For ITT, assume we use the same censoring model (with numerator pooling)
trial_itt['data'] = data_censored.copy()

# ---------------------------------------------------
# 4. CALCULATE WEIGHTS
# ---------------------------------------------------
# For PP, overall weight is the product of switch and censor weights.
trial_pp['data']['weight'] = trial_pp['data']['switch_weight'] * trial_pp['data']['censor_weight']
# For ITT, use censor weight only.
trial_itt['data']['weight'] = trial_itt['data']['censor_weight']

# ---------------------------------------------------
# 5. SPECIFY OUTCOME MODEL
# ---------------------------------------------------
# For ITT, we define the treatment variable and adjustment terms.
# Here, we assume assigned_treatment equals treatment.
trial_itt['data']['assigned_treatment'] = trial_itt['data']['treatment']

# Create additional variables needed for the outcome model:
trial_itt['data']['followup_time'] = trial_itt['data']['period']  # using period as a proxy
trial_itt['data']['followup_time2'] = trial_itt['data']['followup_time'] ** 2
trial_itt['data']['trial_period'] = trial_itt['data']['period']
trial_itt['data']['trial_period2'] = trial_itt['data']['trial_period'] ** 2

# Define outcome model formula:
formula_outcome = ("outcome ~ assigned_treatment + x2 + followup_time + followup_time2 " +
                   "+ trial_period + trial_period2")

# Fit weighted logistic regression using GLM (for ITT)
y_out, X_out = dmatrices(formula_outcome, trial_itt['data'], return_type='dataframe')
weights = trial_itt['data']['weight']
outcome_model = sm.GLM(y_out, X_out, family=sm.families.Binomial(), var_weights=weights).fit()

print("\nOutcome Model Summary (ITT):")
print(outcome_model.summary())

# ---------------------------------------------------
# 6. EXPAND TRIALS
# ---------------------------------------------------
# The R code “expands” the data into a sequence of trials.
# Here we create a function that replicates each patient’s row over a range of followup times.
def expand_trials(data, max_followup=10):
    expanded = []
    for _, row in data.iterrows():
        for t in range(0, max_followup + 1):
            new_row = row.copy()
            new_row['trial_period'] = t
            new_row['followup_time'] = t
            new_row['followup_time2'] = t ** 2
            expanded.append(new_row)
    return pd.DataFrame(expanded)

expanded_data_itt = expand_trials(trial_itt['data'], max_followup=10)

# ---------------------------------------------------
# 7. LOAD OR SAMPLE FROM EXPANDED DATA
# ---------------------------------------------------
# For large datasets, we might sample control (outcome==0) observations.
# Here, we sample outcome==0 with probability p_control=0.5 while keeping all outcome==1 rows.
p_control = 0.5
np.random.seed(1234)
mask_sample = (expanded_data_itt['outcome'] == 1) | (np.random.rand(len(expanded_data_itt)) < p_control)
sampled_data_itt = expanded_data_itt[mask_sample].copy()

# ---------------------------------------------------
# 8. FIT MARGINAL STRUCTURAL MODEL (MSM)
# ---------------------------------------------------
# Winsorize (clip) weights at the 99th percentile to reduce the influence of extreme values.
q99 = np.quantile(sampled_data_itt['weight'], 0.99)
sampled_data_itt['w_mod'] = np.minimum(sampled_data_itt['weight'], q99)

# Fit the MSM using the same outcome model formula, now with the modified weights.
y_msm, X_msm = dmatrices(formula_outcome, sampled_data_itt, return_type='dataframe')
msm_model = sm.GLM(y_msm, X_msm, family=sm.families.Binomial(),
                   var_weights=sampled_data_itt['w_mod']).fit()

print("\nMarginal Structural Model Summary (ITT MSM):")
print(msm_model.summary())

# ---------------------------------------------------
# 9. INFERENCE: PREDICTION AND PLOTTING
# ---------------------------------------------------
# For illustration, we predict survival probabilities (1 - predicted outcome risk)
# at follow-up times 0 to 10 for those in trial_period == 1.
subset_data = sampled_data_itt[sampled_data_itt['trial_period'] == 1]
predict_times = np.arange(0, 11)

predictions = []
for t in predict_times:
    temp = subset_data.copy()
    temp['followup_time'] = t
    temp['followup_time2'] = t ** 2
    # Keep trial_period fixed at 1 for prediction (with its square computed)
    temp['trial_period'] = 1
    temp['trial_period2'] = 1
    y_pred = msm_model.predict(temp)
    survival = 1 - y_pred  # approximate survival probability
    predictions.append({'followup_time': t, 'survival_prob': np.mean(survival)})

pred_df = pd.DataFrame(predictions)

# Plot the survival curve
plt.figure(figsize=(8, 6))
plt.plot(pred_df['followup_time'], pred_df['survival_prob'], marker='o', label='Survival Probability')
plt.xlabel("Follow-up Time")
plt.ylabel("Survival Probability")
plt.title("Predicted Survival over Time (ITT MSM)")
plt.legend()
plt.grid(True)
plt.show()


Sample of data_censored:
   id  period  treatment  x1        x2  x3        x4  age     age_s  outcome  \
0   1       0          1   1  1.146148   0  0.734203   36  0.083333        0   
1   1       1          1   1  0.002200   0  0.734203   37  0.166667        0   
2   1       2          1   0 -0.481762   0  0.734203   38  0.250000        0   
3   1       3          1   0  0.007872   0  0.734203   39  0.333333        0   
4   1       4          1   1  0.216054   0  0.734203   40  0.416667        0   

   censored  eligible  
0         0         1  
1         0         0  
2         0         0  
3         0         0  
4         0         0  


C:\Users\Alex\AppData\Local\Temp\ipykernel_10972\4277196242.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_censored['prev_treatment'].fillna(data_censored['treatment'], inplace=True)


KeyError: 'switch_weight'